# 09.5 · Hugging Face: pretrained models in three lines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/05-hugging-face/lesson.ipynb)

Read the [README](README.md) first. This notebook downloads four small models (about 1 GB in total, once; they are cached outside the course folder) and puts each to work on ShopWave text.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

In [2]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import torch, time
from transformers import pipeline

device = "mps" if torch.backends.mps.is_available() else (0 if torch.cuda.is_available() else -1)
print("running on", {-1: "CPU", 0: "NVIDIA GPU", "mps": "Apple GPU"}[device])

running on Apple GPU


## 1. Sentiment in three lines

In [3]:
sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                     device=device)
sentiment(["The mug arrived chipped.", "Lovely fairy lights!", "Not bad at all, I'm pleasantly surprised."])

[{'label': 'NEGATIVE', 'score': 0.9932029247283936},
 {'label': 'POSITIVE', 'score': 0.9998766183853149},
 {'label': 'POSITIVE', 'score': 0.9998370409011841}]

The third sentence fooled TF-IDF (09.2) and embeddings (09.3). A Transformer fine-tuned for sentiment reads *not bad* correctly. How does it do on all 2,000 test reviews? It was trained on **film** reviews (the SST-2 dataset), never on these product reviews:

In [4]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

10,000 reviews (50% positive): train 8,000 / test 2,000


,text,label
0,"Great Book, Must read! Ramon found the fabulou...",1
1,A leaker! This is the second generation of the...,0
2,Not For the Professional Cook. Henckels are we...,0
3,Arg! I'd like to see Robert Jordan put to the ...,0
4,"If I could give this minus 5 stars, I'd give i...",0


In [5]:
start = time.time()
preds = sentiment(test["text"].tolist(), batch_size=32, truncation=True, max_length=128)
accuracy = np.mean([(p["label"] == "POSITIVE") == (y == 1) for p, y in zip(preds, test["label"])])
print(f"accuracy on 2,000 test reviews: {accuracy:.1%}  ({time.time() - start:.0f} s)")

accuracy on 2,000 test reviews: 89.0%  (3 s)


Without seeing a single one of our labelled reviews, it beats the best TF-IDF model trained on 8,000 of them (88.4%).

## 2. Tokenisers: what the model actually reads

In [6]:
tok = sentiment.tokenizer
sentence = "ShopWave's retrospot mugs arrived unbroken!"
print(tok.tokenize(sentence))
print(tok(sentence)["input_ids"])
print(f"vocabulary: {tok.vocab_size:,} word pieces")

['shop', '##wave', "'", 's', 'retro', '##sp', '##ot', 'mug', '##s', 'arrived', 'unbroken', '!']
[101, 4497, 16535, 1005, 1055, 22307, 13102, 4140, 14757, 2015, 3369, 29505, 999, 102]
vocabulary: 30,522 word pieces


Unknown words are split into known pieces (*retrospot* → *retro*, *##sp*, *##ot*; the *##* means "continues the previous piece"). 101 and 102 are the special tokens [CLS] and [SEP]. Every Transformer has its own tokeniser and must be used with it.

## 3. Routing support emails with zero-shot classification

ShopWave has no labelled support emails. A **zero-shot** classifier is a model trained on *natural language inference* (does sentence A imply sentence B?). For each topic it asks: does this email imply *"This example is {topic}."*? The topic with the highest score wins. No training on our data.

In [7]:
emails = [
    "Hi, my order 18734 still hasn't arrived and it's been two weeks. Where is it?",
    "I'd like my money back for the lantern, it doesn't fit my shelf.",
    "Is the ceramic teapot dishwasher safe?",
    "The cake stand came smashed and nobody answered my last two emails. Very poor service.",
    "Can you tell me if the fairy lights work outdoors?",
    "Please refund the £24.50 for the cancelled doormat.",
    "Tracking says delivered but there is nothing at my door.",
    "Your courier left the parcel in the rain and everything inside is ruined. Unacceptable.",
]
truth = ["delivery", "refund", "product question", "complaint",
         "product question", "refund", "delivery", "complaint"]
topics = ["delivery", "refund", "product question", "complaint"]

def route(model_name):
    clf = pipeline("zero-shot-classification", model=model_name, device=device)
    start = time.time()
    out = clf(emails, candidate_labels=topics)
    pred = [o["labels"][0] for o in out]
    correct = sum(p == t for p, t in zip(pred, truth))
    return clf, out, pred, correct, time.time() - start

clf, out, pred, correct, secs = route("typeform/distilbert-base-uncased-mnli")
print(f"{correct} of {len(emails)} correct ({secs:.1f} s)\n")
for e, o, t in zip(emails, out, truth):
    mark = "✓" if o["labels"][0] == t else "✗"
    print(f"{mark} {o['labels'][0]:17} {o['scores'][0]:.2f}  (truth: {t:16}) {e[:55]}")

6 of 8 correct (0.2 s)

✗ complaint         0.29  (truth: delivery        ) Hi, my order 18734 still hasn't arrived and it's been t
✗ product question  0.32  (truth: refund          ) I'd like my money back for the lantern, it doesn't fit 
✓ product question  0.97  (truth: product question) Is the ceramic teapot dishwasher safe?
✓ complaint         0.51  (truth: complaint       ) The cake stand came smashed and nobody answered my last
✓ product question  0.79  (truth: product question) Can you tell me if the fairy lights work outdoors?
✓ refund            0.97  (truth: refund          ) Please refund the £24.50 for the cancelled doormat.
✓ delivery          0.82  (truth: delivery        ) Tracking says delivered but there is nothing at my door
✓ complaint         0.71  (truth: complaint       ) Your courier left the parcel in the rain and everything


Six of eight with no training data at all. The mistakes are understandable: a parcel that "still hasn't arrived" reads as a complaint, and the lantern refund, which explains that the lantern does not fit a shelf, reads as a product question. With a few hundred labelled emails, a fine-tuned model (09.6) would do better.

## 4. Pulling out names, places and organisations

In [8]:
ner = pipeline("token-classification", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)
for entity in ner("Sarah Jones from Paris ordered 12 Kew Gardens mugs from ShopWave in London."):
    print(f"{entity['entity_group']:4} {entity['score']:.2f}  {entity['word']}")

PER  1.00  Sarah Jones
LOC  1.00  Paris
LOC  0.67  Kew Gardens
ORG  0.99  ShopWave
LOC  1.00  London


PER = person, LOC = location, ORG = organisation. *Kew Gardens* is tagged as a place (it is one, but here it is a product range), with lower confidence. Useful for spotting personal data in emails before storing them.

## 5. What BERT was trained to do: fill in the blank

In [9]:
fill = pipeline("fill-mask", model="distilbert/distilbert-base-uncased", device=device)
for r in fill("The parcel arrived [MASK] and the mug was broken."):
    print(f"{r['score']:.3f}  {r['token_str']}")

0.132  safely
0.101  promptly
0.069  quickly
0.043  immediately
0.033  soon


BERT-style models are pre-trained by hiding words and guessing them. It fills the blank with *safely*, the most likely word in general text, even though the rest of the sentence says the mug broke. It predicts plausible words, not facts. Generative LLMs (09.7) are trained the same way, one next word at a time, and inherit the same habit.

## 6. What does a model cost?

In [10]:
models = {"sentiment (DistilBERT)": sentiment.model, "zero-shot (DistilBERT-MNLI)": clf.model,
          "NER (BERT-base)": ner.model, "fill-mask (DistilBERT)": fill.model}
rows = []
for name, m in models.items():
    n = sum(p.numel() for p in m.parameters())
    rows.append({"model": name, "parameters (millions)": round(n / 1e6, 1),
                 "memory float32 (MB)": round(n * 4 / 1e6), "memory float16 (MB)": round(n * 2 / 1e6)})
pd.DataFrame(rows)

,model,parameters (millions),memory float32 (MB),memory float16 (MB)
0,sentiment (DistilBERT),67.0,268,134
1,zero-shot (DistilBERT-MNLI),67.0,268,134
2,NER (BERT-base),107.7,431,215
3,fill-mask (DistilBERT),67.0,268,134


Every parameter is a 4-byte number (float32) or a 2-byte one (float16). A model's memory is roughly *parameters × bytes per parameter*, plus working space. These are small: a large language model has billions of parameters (09.7), which is why the [setup guide](../../../setup/README.md#running-a-model-on-your-own-computer-with-ollama) matches models to your RAM.

## Quiz answers

1. A tokeniser, the model itself, and the pre- and post-processing that turn text into model inputs and model outputs into labels and scores.
2. Because a model only understands the token ids it was trained with; another tokeniser would produce ids that mean something different to it.
3. It was trained on natural language inference, so it can score whether the text implies "This example is {label}." for any label you write, without training on your labels.
4. About 66 million × 4 bytes ≈ 265 MB in float32, half that in float16.
5. It was pre-trained to fill in hidden words with the most *likely* word in general text, not to check facts; *safely* is common after "arrived".

**Next:** [09.6 Fine-tuning](../06-fine-tuning/)